# Week 09A — Data-Centric Machine Vision

**MCTA 4364 Machine Vision** · Session 1 of Week 9 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W09_data_centric/W09A_data_centric_machine_vision.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W09_data_centric/W09A_data_centric_machine_vision.ipynb)

In industry, the model architecture is rarely what decides success. Two teams using the same YOLO or ResNet get very different results because
of their **data**: how consistently it is labelled, whether it covers the lighting and poses of the real line, how rare defects are handled, and
whether the test set is trustworthy. **Data-centric** machine vision treats the dataset as the thing you engineer, while the model code stays fixed.

In this session the model is **frozen** (the same small CNN every time). You change only the data, and measure the effect of each change.

### Learning outcomes
By the end of this session you will be able to:
1. Explain the model-centric vs data-centric workflow and the **data flywheel**.
2. Show why **systematic** label errors hurt far more than random ones, and why noisy **test** labels corrupt evaluation.
3. Find likely label errors with **out-of-fold predictions** (confident learning) and prioritise human review.
4. Handle **class imbalance** and **coverage gaps** with re-sampling and targeted augmentation.
5. Audit a dataset: near-duplicates and **leakage**, box sanity checks, labelling guidelines.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–8 | 1–2 | Model-centric vs data-centric; our dataset |
| 8–30 | 3–5 | Label noise: random vs systematic; noisy test sets; finding label errors |
| 30–45 | 6–7 | Class imbalance; coverage and augmentation |
| 45–60 | 8–9 | Dataset hygiene: near-duplicates, box sanity checks |
| 60–72 | 10 | The data-centric workflow: guidelines, gold sets, tools |
| 72–80 | 13 | Quiz and exit ticket |
| Home | 11–12 | Exercises with self-checks, challenge |

**Hardware:** about 2–3 minutes on a laptop CPU (11 short trainings of a 25 k-parameter CNN); seconds on a GPU. No downloads.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset, WeightedRandomSampler

import partsim
from cvhelpers import SMOKE, check, interact, pipeline_diagram, quiz, show
from dlhelpers import confusion_matrix, count_params, fit, hw_profile, plot_confusion, predict, set_seed

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = HW["device"]
NAMES = partsim.SCENE_CLASSES
WASHER, FLANGE = NAMES.index("washer"), NAMES.index("flange")
N_PER_CLASS, N_TEST, EPOCHS = (40, 20, 1) if SMOKE else (400, 150, 8)
t0 = time.perf_counter()
X_train, y_train = partsim.make_crops(N_PER_CLASS, seed=1)                        # 48 x 48 greyscale crops
X_test, y_test = partsim.make_crops(N_TEST, seed=2)
X_shift, y_shift = partsim.make_crops(N_TEST, seed=3, exposure=0.55, noise=6.0)   # a darker, noisier camera
print(f"{len(y_train)} training, {len(y_test)} test crops ({time.perf_counter() - t0:.0f} s)")

## 1. Model-centric vs data-centric

| | model-centric | data-centric |
|---|---|---|
| fixed | the dataset | the model and training code |
| iterate on | architecture, hyper-parameters | labels, coverage, balance, cleaning, augmentation |
| typical gain on a mature problem | small | often large |
| key question | *which model is best?* | *what is wrong with my data, and which fix is cheapest?* |

In deployed systems the dataset keeps growing: errors in production are captured, labelled and added — the **data flywheel**.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.6))
pipeline_diagram(["collect\n(cover the line)", "label with\nguidelines", "audit\n(noise, balance, leaks)", "train\n(fixed model)",
                  "error analysis\n(per class, per slice)", "deploy +\nmonitor", "capture failures\n-> back to collect"],
                 colors=["#e8f0fe", "#e8f0fe", "#fce8e6", "#f1f3f4", "#fef7e0", "#e6f4ea", "#e6f4ea"], ax=ax, fontsize=10)
ax.set_title("Figure 1.1 - The data flywheel: most of the engineering effort sits outside the 'train' box", fontweight="bold")
plt.show()

## 2. Our dataset and our frozen model

We classify **crops of single parts** from the conveyor simulator of Week 8 into 4 classes. Washers and flanges are both round with a hole:
they are the classes most likely to be confused, by people and by models. The model below is fixed for the whole session.

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(20, 8.4))
for c in range(4):
    idx = np.nonzero(y_train == c)[0][:10]
    for k, i in enumerate(idx):
        axes[c, k].imshow(X_train[i], cmap="gray", vmin=0, vmax=255); axes[c, k].set_xticks([]); axes[c, k].set_yticks([])
    axes[c, 0].set_ylabel(NAMES[c], fontsize=12)
plt.suptitle("Figure 2.1 - Training crops (48 x 48): note partial views, rotation, lighting gradients", fontweight="bold")
plt.tight_layout(); plt.show()


def to_tensor(X):
    return torch.tensor(np.asarray(X)[:, None] / 255.0, dtype=torch.float32)


def frozen_model():
    """The model we never change in this session."""
    def block(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(block(1, 16), block(16, 32), block(32, 64), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 4))


def train(dataset, sampler=None, epochs=EPOCHS, seed=0):
    set_seed(seed)
    model = frozen_model().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), 3e-3)
    tr = DataLoader(dataset, batch_size=64, shuffle=sampler is None, sampler=sampler)
    va = DataLoader(TensorDataset(to_tensor(X_test), torch.tensor(y_test)), batch_size=256)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, total_steps=epochs * len(tr))     # a schedule makes runs repeatable
    fit(model, tr, va, opt, epochs, DEVICE, scheduler=sched, verbose=False)
    return model


def evaluate(model, X, y):
    logits, labels = predict(model, DataLoader(TensorDataset(to_tensor(X), torch.tensor(y)), batch_size=256), DEVICE)
    pred = logits.argmax(1)
    cm = confusion_matrix(labels, pred, 4)
    return dict(acc=float((pred == labels).mean()), recall=cm.diagonal() / cm.sum(1).clip(min=1), cm=cm, probs=torch.softmax(torch.tensor(logits), 1).numpy())


def ds(X, y):
    return TensorDataset(to_tensor(X), torch.tensor(y))


print(f"frozen model: {count_params(frozen_model()):,} parameters")
t0 = time.perf_counter()
m_clean = train(ds(X_train, y_train))
r_clean = evaluate(m_clean, X_test, y_test)
print(f"clean labels: test accuracy {r_clean['acc']:.1%}  (trained in {time.perf_counter() - t0:.0f} s)")

## 3. Label noise: random vs systematic

Two kinds of labelling mistakes:
- **random** noise: a tired annotator occasionally clicks a random wrong class;
- **systematic** (class-conditional) noise: the guideline is ambiguous, so an annotator *consistently* calls some washers "flange".

We corrupt the labels in each way and retrain the frozen model: 30% of **all** labels at random, or 35% of the **washers** systematically.

In [ ]:
rng = np.random.default_rng(0)
y_random = y_train.copy()
flip = rng.random(len(y_random)) < 0.30
y_random[flip] = (y_random[flip] + rng.integers(1, 4, flip.sum())) % 4          # a different, random class

y_system = y_train.copy()
is_flipped = (y_train == WASHER) & (rng.random(len(y_train)) < 0.35)
y_system[is_flipped] = FLANGE                                                    # washers called 'flange'
print(f"random noise: {(y_random != y_train).mean():.1%} of all labels wrong | "
      f"systematic noise: {is_flipped.sum()} washers labelled 'flange' ({(y_system != y_train).mean():.1%} of all labels)")

m_random = train(ds(X_train, y_random)); r_random = evaluate(m_random, X_test, y_test)
m_system = train(ds(X_train, y_system)); r_system = evaluate(m_system, X_test, y_test)

fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), gridspec_kw=dict(width_ratios=[1.3, 1, 1, 1]))
names_ = ["clean", "30% random", "35% of washers\nlabelled flange"]
res_ = [r_clean, r_random, r_system]
xs = np.arange(3)
axes[0].bar(xs - 0.2, [r["acc"] for r in res_], 0.4, label="overall accuracy", color="#1a73e8")
axes[0].bar(xs + 0.2, [r["recall"][WASHER] for r in res_], 0.4, label="washer recall", color="#f29900")
axes[0].set_xticks(xs); axes[0].set_xticklabels(names_); axes[0].set_ylim(0, 1.05); axes[0].legend(loc="lower left"); axes[0].grid(alpha=0.3, axis="y")
for ax, r, n in zip(axes[1:], res_, names_):
    plot_confusion(r["cm"], NAMES, title=n.replace("\n", " "), ax=ax)
plt.suptitle("Figure 3.1 - Same model, same images, different labels (clean test set)", fontweight="bold")
plt.tight_layout(); plt.show()

**Why?** Random errors point in all directions and largely **average out**: the network still sees the correct class as the most frequent label for
each kind of image (deep networks are known to be robust to large amounts of *random* noise; Rolnick et al., 2017). Systematic errors all point
the **same way**, so the network faithfully learns the annotator's bias. Real labelling errors are mostly systematic: unclear class boundaries, tiny
or occluded objects, look-alike classes.

## 4. Noisy **test** labels corrupt your metrics

If the **test** set was labelled by the same biased process, your evaluation is wrong too. Northcutt et al. (2021) estimated that about 3.4% of the
labels in the test sets of ten popular benchmarks (ImageNet, CIFAR, QuickDraw...) are wrong — enough to change which model looks best.

Below, the *clean* model is evaluated against a test set with the same systematic washer → flange errors.

In [ ]:
y_test_noisy = y_test.copy()
bad = (y_test == WASHER) & (np.random.default_rng(5).random(len(y_test)) < 0.30)
y_test_noisy[bad] = FLANGE
r_true = evaluate(m_clean, X_test, y_test)
r_reported = evaluate(m_clean, X_test, y_test_noisy)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_confusion(r_true["cm"], NAMES, title=f"true labels: accuracy {r_true['acc']:.1%}", ax=axes[0])
plot_confusion(r_reported["cm"], NAMES, title=f"noisy test labels: reported {r_reported['acc']:.1%}", ax=axes[1])
plt.suptitle("Figure 4.1 - The same predictions, scored against clean and against noisy test labels", fontweight="bold")
plt.tight_layout(); plt.show()
print("The model did not change; only the answer key did. Test and validation labels deserve the MOST care (double labelling, expert review).")

## 5. Finding label errors: out-of-fold predictions and confident learning

Idea (*confident learning*, Northcutt et al., 2021; the `cleanlab` library): a model that has **not seen** an example during training gives an honest
opinion about it. If it confidently disagrees with the given label, the label is suspicious.

1. Split the training set into $K$ folds; train on $K-1$ folds and predict the held-out fold → **out-of-fold** probabilities $\hat{p}(c \mid x)$ for
   every training image.
2. **Self-confidence** of an example = $\hat{p}(\text{given label} \mid x)$. Low self-confidence = likely label error.
3. Send the lowest-ranked examples to a **human reviewer** (never relabel automatically: the model is wrong sometimes too).

**What the top of the list shows.** The very first suspects are often *correctly* labelled but **ambiguous** crops: parts cut by the crop border
or touching a neighbour. Reviewing them is still valuable: they are exactly the edge cases your labelling guideline must settle.

**Limitation.** Confident learning assumes that, for every class, the correct labels are the **majority**. If an annotator mislabels more than about
40% of a class, the out-of-fold models learn the bias too, and the "suspicious" examples become the *correct* labels that disagree with it (try it:
change 0.35 to 0.5 in Section 3). Systematic errors of that size must be prevented upstream, with guidelines and gold sets (Section 10).

In [ ]:
def out_of_fold_probs(X, y, k=3, seed=0):
    idx = np.random.default_rng(seed).permutation(len(y))
    probs = np.zeros((len(y), 4), np.float32)
    for fold in np.array_split(idx, k):
        train_idx = np.setdiff1d(np.arange(len(y)), fold)
        m = train(ds(X[train_idx], y[train_idx]))
        probs[fold] = evaluate(m, X[fold], y[fold])["probs"]
    return probs


t0 = time.perf_counter()
P_oof = out_of_fold_probs(X_train, y_system)
print(f"out-of-fold probabilities for {len(y_system)} images in {time.perf_counter() - t0:.0f} s")
self_conf = P_oof[np.arange(len(y_system)), y_system]
ranking = np.argsort(self_conf)                                      # most suspicious first

fig, axes = plt.subplots(2, 10, figsize=(20, 5))
for k, ax in enumerate(axes.ravel()):
    i = ranking[k]
    ax.imshow(X_train[i], cmap="gray", vmin=0, vmax=255); ax.set_xticks([]); ax.set_yticks([])
    col = "#d93025" if is_flipped[i] else "#188038"
    ax.set_title(f"given: {NAMES[y_system[i]]}\nmodel: {NAMES[P_oof[i].argmax()]} ({P_oof[i].max():.2f})", fontsize=8, color=col)
plt.suptitle("Figure 5.1 - The 20 most suspicious labels (red title = a label we corrupted, green = a correct label the model doubts)",
             fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
budget = np.arange(0, 3 * max(is_flipped.sum(), 1) + 1)
found_ranked = np.r_[0, np.cumsum(is_flipped[ranking])][budget]
found_random = budget * is_flipped.mean()
review = ranking[:2 * is_flipped.sum()]
y_reviewed = y_system.copy()
y_reviewed[review] = y_train[review]                  # the human reviewer restores the true label of every reviewed image
m_reviewed = train(ds(X_train, y_reviewed)); r_reviewed = evaluate(m_reviewed, X_test, y_test)

fig, axes = plt.subplots(1, 2, figsize=(17, 4.8))
axes[0].plot(budget, found_ranked, lw=2.5, label="review in order of self-confidence")
axes[0].plot(budget, found_random, lw=2.5, ls="--", label="review a random sample")
axes[0].axhline(is_flipped.sum(), color="k", lw=1, ls=":"); axes[0].axvline(len(review), color="#d93025", lw=1, ls=":")
axes[0].set_xlabel("number of images reviewed by a human"); axes[0].set_ylabel("label errors found"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title("ranking makes human review far more efficient")
names_ = ["clean", "35% of washers\nlabelled flange", "after reviewing\nthe top-ranked"]
res_ = [r_clean, r_system, r_reviewed]
axes[1].bar(np.arange(3) - 0.2, [r["acc"] for r in res_], 0.4, label="accuracy", color="#1a73e8")
axes[1].bar(np.arange(3) + 0.2, [r["recall"][WASHER] for r in res_], 0.4, label="washer recall", color="#f29900")
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(names_); axes[1].set_ylim(0, 1.05); axes[1].legend(loc="lower left"); axes[1].grid(alpha=0.3, axis="y")
axes[1].set_title(f"retrained after reviewing {len(review)} images ({len(review) / len(y_train):.0%} of the set)")
plt.suptitle("Figure 5.2 - Human-in-the-loop label cleaning", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"errors found in the reviewed {len(review)} images: {is_flipped[review].sum()} of {is_flipped.sum()} "
      f"(a random sample of the same size would contain about {len(review) * is_flipped.mean():.0f})")

## 6. Class imbalance

Defects and rare variants are, by definition, rare. With 400 examples of each class but only **10 washers**, the model rarely sees a washer and
drifts towards the look-alike flange. Remedies, cheapest first:
1. **collect more** of the rare class (targeted data collection, synthetic data in Week 13B);
2. **re-sample**: draw rare examples more often (`WeightedRandomSampler`), or **re-weight** the loss by inverse class frequency;
3. use a loss that focuses on hard examples (**focal loss**), and pick the decision threshold per class on a validation set.

Always report **per-class recall**; overall accuracy hides the problem when the rare class is rare in the test set too.

In [ ]:
keep = np.r_[np.nonzero(y_train != WASHER)[0], np.nonzero(y_train == WASHER)[0][:10]]
X_imb, y_imb = X_train[keep], y_train[keep]
print("class counts:", dict(zip(NAMES, np.bincount(y_imb, minlength=4))))
m_imb = train(ds(X_imb, y_imb)); r_imb = evaluate(m_imb, X_test, y_test)
weights = 1.0 / np.bincount(y_imb)[y_imb]                          # each class drawn equally often
m_bal = train(ds(X_imb, y_imb), sampler=WeightedRandomSampler(weights, num_samples=len(y_imb), replacement=True))
r_bal = evaluate(m_bal, X_test, y_test)
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8), gridspec_kw=dict(width_ratios=[1.2, 1, 1]))
xs = np.arange(4)
for k, (r, n, col) in enumerate([(r_clean, "balanced data", "#9aa0a6"), (r_imb, "only 10 washers", "#d93025"), (r_bal, "10 washers, oversampled", "#188038")]):
    axes[0].bar(xs + (k - 1) * 0.27, r["recall"], 0.27, label=n, color=col)
axes[0].set_xticks(xs); axes[0].set_xticklabels(NAMES); axes[0].set_ylabel("recall"); axes[0].set_ylim(0, 1.05); axes[0].legend(loc="lower left")
axes[0].grid(alpha=0.3, axis="y"); axes[0].set_title("per-class recall")
plot_confusion(r_imb["cm"], NAMES, title="only 10 washers", ax=axes[1])
plot_confusion(r_bal["cm"], NAMES, title="oversampled", ax=axes[2])
plt.suptitle("Figure 6.1 - Class imbalance and re-sampling (balanced test set)", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Coverage: the data must span the operating conditions

The training images were all taken under nominal lighting. On the line, the exposure drifts (Week 7). A test set collected with a **darker, noisier**
camera measures how well the data **cover** those conditions.

**Augmentation** is a data-centric tool: it encodes what you know about the variation the model must ignore. Here: brightness and contrast changes,
sensor noise, and 90° rotations (parts lie in any orientation). Only augment with changes that preserve the label (a mirror image of a text label
is not the same label!). Augmented data make the task harder, so we train for twice as many epochs.

In [ ]:
class Augmented(Dataset):
    """Photometric and rotation augmentation applied on the fly (a new random version every epoch)."""
    def __init__(self, X, y):
        self.X, self.y = to_tensor(X), torch.tensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.X[i]
        gain, contrast, noise = np.random.uniform(0.45, 1.2), np.random.uniform(0.7, 1.3), np.random.uniform(0, 0.035)
        m = x.mean()
        x = ((x - m) * contrast + m) * gain + torch.randn_like(x) * noise
        x = torch.rot90(x, int(np.random.randint(4)), (1, 2))
        return x.clamp(0, 1), self.y[i]


aug_view = Augmented(X_train[:1].repeat(8, 0), y_train[:1].repeat(8))
np.random.seed(0)
show(*[aug_view[i][0][0].numpy() for i in range(8)], titles=[f"augmented #{i}" for i in range(8)], cmap="gray", figsize=(20, 3))
m_aug = train(Augmented(X_train, y_train), epochs=2 * EPOCHS)       # augmented data are a harder task: train longer
rows = [("no augmentation", m_clean), ("photometric + rotation augmentation", m_aug)]
res_cov = {n: (evaluate(m, X_test, y_test)["acc"], evaluate(m, X_shift, y_shift)["acc"]) for n, m in rows}
fig, axes = plt.subplots(1, 2, figsize=(17, 4.5), gridspec_kw=dict(width_ratios=[1, 1.2]))
xs = np.arange(2)
axes[0].bar(xs - 0.2, [v[0] for v in res_cov.values()], 0.4, label="nominal test set", color="#1a73e8")
axes[0].bar(xs + 0.2, [v[1] for v in res_cov.values()], 0.4, label="darker, noisier camera", color="#f29900")
axes[0].set_xticks(xs); axes[0].set_xticklabels(list(res_cov)); axes[0].set_ylim(0, 1.05); axes[0].legend(loc="lower left"); axes[0].grid(alpha=0.3, axis="y")
axes[1].imshow(np.hstack([np.hstack(list(X_test[:6])), np.hstack(list(X_shift[:6]))]), cmap="gray", vmin=0, vmax=255); axes[1].axis("off")
axes[1].set_title("nominal test crops (left) vs shifted camera (right)")
plt.suptitle("Figure 7.1 - Covering the operating conditions with targeted augmentation", fontweight="bold")
plt.tight_layout(); plt.show()
for n, (a, b) in res_cov.items():
    print(f"{n:<40} nominal {a:.1%} | shifted camera {b:.1%}")

## 8. Hygiene I: near-duplicates and leakage

Video frames, bursts of photos and re-exports create **near-duplicate** images. If copies of the same scene end up in both the training and the test
split, the test accuracy measures memory, not generalisation (**leakage**). Always split by *part / batch / day / camera*, not by image, and check
for duplicates.

A cheap and robust detector: shrink each image to a tiny **thumbnail signature** (32 × 24 grey levels), subtract its mean and normalise it to unit
length. The dot product of two signatures is their **correlation** (1 = identical up to brightness and contrast). *Perceptual hashes* (dHash, pHash)
and CNN embeddings (DINOv2, CLIP) are alternatives; hashes are fragile on large flat areas such as our conveyor belt, where noise flips their bits.

In [ ]:
def signature_ref(img, w=32, h=24):
    g = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (w, h), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    g -= g.mean()
    return g / (np.linalg.norm(g) + 1e-9)


# simulate a careless split of a scene dataset: 10 "test" images are re-exported copies of training images
rng = np.random.default_rng(11)
n_tr, n_te = (20, 8) if SMOKE else (80, 30)
scenes_tr = [partsim.make_scene(rng.integers(1 << 31))[0] for _ in range(n_tr)]
scenes_te = [partsim.make_scene(rng.integers(1 << 31))[0] for _ in range(n_te)]
leaked_src = rng.choice(n_tr, 10, replace=False)
copies = [np.clip(scenes_tr[i].astype(float) * rng.uniform(0.8, 1.2) + rng.normal(0, 3, scenes_tr[i].shape), 0, 255).astype(np.uint8)
          for i in leaked_src]
test_all = scenes_te + copies
sim = np.array([signature_ref(x) for x in test_all]) @ np.array([signature_ref(x) for x in scenes_tr]).T
best = sim.max(1)
fig, axes = plt.subplots(1, 2, figsize=(18, 4.5), gridspec_kw=dict(width_ratios=[1, 1.3]))
bins = np.linspace(-0.2, 1.0, 61)
axes[0].hist(best[:n_te], bins, alpha=0.7, label="genuine test images", color="#1a73e8")
axes[0].hist(best[n_te:], bins, alpha=0.8, label="re-exported copies of training images", color="#d93025")
axes[0].axvline(0.95, color="k", ls="--"); axes[0].set_xlabel("highest correlation with any training image"); axes[0].legend(loc="upper left")
flag = np.nonzero(best > 0.95)[0]
axes[0].set_title(f"threshold 0.95: {np.sum(flag >= n_te)}/10 copies flagged, {np.sum(flag < n_te)} genuine images flagged")
pairs = [np.hstack([test_all[i], scenes_tr[sim[i].argmax()]]) for i in flag[:2]]
axes[1].imshow(np.vstack(pairs)[..., ::-1] if pairs else np.zeros((10, 10))); axes[1].axis("off")
axes[1].set_title("flagged pairs: test image | its most similar training image")
plt.suptitle("Figure 8.1 - Catching train/test leakage with thumbnail signatures", fontweight="bold")
plt.tight_layout(); plt.show()

## 9. Hygiene II: auditing detection labels

Detection labels have their own failure modes: boxes outside the image, zero-size or tiny boxes, exact **duplicates** (double-clicked), boxes with the
wrong class, loose boxes, and **missing** boxes. Most can be caught automatically *before* training (exercise E3). Below are the label statistics of 30 generated scenes — the same plots YOLO saves
as `labels.jpg` when it starts training. Outliers in these plots are where to look first.

In [ ]:
rng = np.random.default_rng(7)
scenes = [partsim.make_scene(rng.integers(1 << 31)) for _ in range(10 if SMOKE else 30)]
all_boxes = np.concatenate([b for _, b, _ in scenes]); all_labels = np.concatenate([l for _, _, l in scenes])
wh = np.c_[all_boxes[:, 2] - all_boxes[:, 0], all_boxes[:, 3] - all_boxes[:, 1]]
fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
axes[0].bar(NAMES, np.bincount(all_labels, minlength=4), color=plt.cm.tab10.colors[:4]); axes[0].set_title("instances per class")
for c in range(4):
    axes[1].scatter(wh[all_labels == c, 0], wh[all_labels == c, 1], s=10, label=NAMES[c], color=plt.cm.tab10.colors[c])
axes[1].set_xlabel("box width (px)"); axes[1].set_ylabel("box height (px)"); axes[1].legend(); axes[1].set_title("box sizes: clusters per class")
centres = np.c_[(all_boxes[:, 0] + all_boxes[:, 2]) / 2, (all_boxes[:, 1] + all_boxes[:, 3]) / 2]
axes[2].hist2d(centres[:, 0], centres[:, 1], bins=[16, 12], range=[[0, 640], [0, 480]], cmap="Blues"); axes[2].invert_yaxis()
axes[2].set_title("where objects are (check: no blind areas)")
plt.suptitle("Figure 9.1 - Label statistics of a detection dataset", fontweight="bold")
plt.tight_layout(); plt.show()

## 10. The data-centric workflow in practice

**A labelling guideline** (one page, with pictures) is the single most effective tool. For our conveyor it would say, for example:
- *washer*: flat ring, **no bolt holes**; *flange*: disc with **four bolt holes and a notch**. When the holes are hidden, label by the notch.
- Label an object if **at least 50% is visible** (this is exactly the rule `partsim` uses); draw the box around the **visible** part.
- Boxes must be **tight** (within 2 px); never label reflections or shadows.
- When unsure, use the class `unsure` (reviewed later) — never guess.

**Measure label quality**: have two people label the same 50 images and compute their agreement (IoU of boxes, Cohen's κ of classes). Keep a small
**gold set** labelled by experts to test both annotators and models.

**Tools**: CVAT, Label Studio, Roboflow (labelling); FiftyOne, cleanlab (finding errors, duplicates, gaps); DVC or dataset versions (reproducibility);
*datasheets for datasets* (documentation). Week 13A shows **auto-labelling** with foundation models to speed all of this up.

## 11. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Rank label issues
Given out-of-fold probabilities `probs` `[N, C]` and the given `labels` `[N]`, return the indices of all examples sorted from **most** to **least** suspicious by self-confidence $\hat p(\text{given label} \mid x)$.

In [ ]:
def rank_label_issues(probs, labels):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    p = np.array([[0.9, 0.1], [0.2, 0.8], [0.6, 0.4], [0.05, 0.95]])
    order = list(np.asarray(rank_label_issues(p, np.array([0, 0, 1, 1]))).ravel())
    assert [int(v) for v in order] == [1, 2, 0, 3], f"expected [1, 2, 0, 3], got {order}"
    r = np.asarray(rank_label_issues(P_oof, y_system))
    assert sorted(r.tolist()) == list(range(len(y_system))), "must return a permutation of all indices"
    if not SMOKE:
        top = r[:2 * is_flipped.sum()]
        assert is_flipped[top].mean() > 2 * is_flipped.mean(), "the ranking should find label errors much faster than random review"


check("E1 rank_label_issues", _test_e1)

### E2. Thumbnail signature
Implement `signature(img, w=32, h=24)` of Section 8 for a **BGR** image: convert to grey, resize to `(w, h)` with `INTER_AREA`, flatten to float32, subtract the mean and divide by the norm. Do not call `signature_ref`.

In [ ]:
def signature(img, w=32, h=24):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    v = np.asarray(signature(scenes_tr[0]), np.float32)
    assert v.shape == (32 * 24,), f"expected a vector of 768 numbers, got {v.shape}"
    assert abs(np.linalg.norm(v) - 1) < 1e-4 and abs(v.mean()) < 1e-4, "the signature must have zero mean and unit norm"
    assert np.allclose(v, signature_ref(scenes_tr[0]), atol=1e-5), "differs from the reference"
    brighter = np.clip(scenes_tr[0].astype(float) * 1.15, 0, 255).astype(np.uint8)
    assert float(v @ np.asarray(signature(brighter))) > 0.99, "a brighter copy should still correlate > 0.99"
    assert float(v @ np.asarray(signature(scenes_tr[1]))) < 0.9, "two different scenes should not look like duplicates"


check("E2 signature", _test_e2)

### E3. Box sanity checks
Write `audit_boxes(boxes, labels, img_w, img_h, min_side=4, dup_iou=0.9)` that returns a **dict** `{box_index: reason}` using these reasons, checked in
this order: `"out_of_bounds"` (any coordinate outside the image by more than 1 px), `"invalid"` (x_max ≤ x_min or y_max ≤ y_min),
`"too_small"` (width or height < `min_side`), `"duplicate"` (IoU ≥ `dup_iou` with an **earlier** box of the same class).

In [ ]:
def audit_boxes(boxes, labels, img_w, img_h, min_side=4, dup_iou=0.9):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    img, b, l = scenes[0]
    assert audit_boxes(b, l, 640, 480) == {}, "clean generated labels should pass"
    bad_b = np.vstack([b, [[600, 400, 700, 470]], [[50, 50, 40, 90]], [[10, 10, 12, 30]], b[:1] + 0.5])
    bad_l = np.r_[l, 0, 1, 2, l[0]]
    n = len(b)
    got = audit_boxes(bad_b, bad_l, 640, 480)
    want = {n: "out_of_bounds", n + 1: "invalid", n + 2: "too_small", n + 3: "duplicate"}
    assert got == want, f"expected {want}, got {got}"


check("E3 audit_boxes", _test_e3)

### E4. How much data do you need? (no self-check)
Train the frozen model on 10, 25, 50, 100, 200 and 400 crops per class **with** augmentation, using the same number of gradient steps for each
run, and plot the test accuracy on the **shifted** camera vs the dataset size (log scale), averaged over 3 seeds. Where does the curve flatten?
What would you do next: more data, better data, or a different model?

In [ ]:
# E4: your code here

## 12. Challenge: active learning
You have 2000 unlabelled crops and budget to label 200. Start from 40 labelled crops; in 4 rounds, label 40 more chosen either **at random** or by
**uncertainty** (highest predictive entropy of the current model), retrain, and plot accuracy vs labels used for both strategies (3 seeds).
Then combine uncertainty with **diversity** (e.g. pick from different k-means clusters of the embeddings). When does active learning *not* help?

In [ ]:
# Your challenge code here

## 13. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'In a data-centric project, what is held fixed?',
        "options": [
            'The dataset',
            'The model and training code',
            'Nothing',
            'The test set only',
        ],
        "answer": '130953aa57',
        "explain": 'You iterate on the data and measure the effect with a fixed model.',
    },
    {
        "q": 'Which label errors usually hurt a deep network most?',
        "options": [
            'Random errors',
            'Systematic (class-conditional) errors',
            'Errors in easy images',
            'None',
        ],
        "answer": '0998edc723',
        "explain": 'Consistent errors are learned as if they were the truth.',
    },
    {
        "q": 'Why must out-of-fold predictions be used to find label errors?',
        "options": [
            'They are faster',
            'A model memorises its own training labels, so it cannot judge them',
            'They use more data',
            'They avoid augmentation',
        ],
        "answer": '261ef82d3a',
        "explain": 'Only held-out predictions are an honest second opinion.',
    },
    {
        "q": 'Your test set has systematic label errors. What happens to the reported accuracy?',
        "options": [
            'It is exact',
            'It is wrong, and can even rank models incorrectly',
            'It always increases',
            'Nothing, only training is affected',
        ],
        "answer": '616e2c33bf',
        "explain": 'The metric is only as good as the answer key.',
    },
    {
        "q": 'You have 10 examples of a rare defect and 4000 good parts. Which is NOT a sensible first step?',
        "options": [
            'Collect more defect images',
            'Oversample the defect class',
            'Report only overall accuracy',
            'Look at per-class recall',
        ],
        "answer": '4ef7c17aa8',
        "explain": '99.8% accuracy can mean every defect is missed.',
    },
    {
        "q": 'Frames 1-100 of a video go to training and frames 101-110 to testing. What is the main risk?',
        "options": [
            'Class imbalance',
            'Leakage through near-duplicate frames',
            'Too little data',
            'Label noise',
        ],
        "answer": '125c67ec62',
        "explain": 'Split by video, part or day, not by frame.',
    },
    {
        "q": 'Which augmentation is NOT label-preserving for reading printed part numbers?',
        "options": [
            'Brightness change',
            'Small blur',
            'Horizontal mirror',
            'Noise',
        ],
        "answer": '0658de4592',
        "explain": 'Mirrored text is not the same text.',
    },
    {
        "q": 'What is the most effective single tool for consistent labels?',
        "options": [
            'A bigger model',
            'A written labelling guideline with examples',
            'More GPUs',
            'Lower learning rate',
        ],
        "answer": 'd364ac7141',
        "explain": 'Ambiguity in the guideline becomes systematic noise.',
    },
], title='Week 09A quiz')

## 14. Exit ticket (reflection)
1. Which single data problem from today do you expect in your own project, and how will you detect it?
2. Write one line of a labelling guideline for your project that removes an ambiguity.
3. How will you split your data so that the test set measures generalisation, not memory?

## 15. What's new (2025–26)

> - **Foundation-model-assisted labelling** (SAM 2, Grounding DINO, VLMs; Week 13A) makes the *first* labels cheap; data-centric review makes them *right*.
> - **Embedding-based data exploration** (FiftyOne Brain, Lightly, cleanlab) finds duplicates, outliers, label errors and under-represented slices using
>   features from models such as DINOv2 and CLIP.
> - **Synthetic data** from simulators and diffusion models (Week 13B) fills coverage gaps for rare defects and conditions; the data-centric question
>   is how to mix and validate it.
> - Industrial platforms (Landing AI, MVTec, Roboflow) now centre their workflow on label consistency and data versioning rather than on model choice.

## 16. References and further exploration
- Northcutt, C., Jiang, L. & Chuang, I. (2021). *Confident learning: Estimating uncertainty in dataset labels.* JAIR 70. [arXiv:1911.00068](https://arxiv.org/abs/1911.00068)
- Northcutt, C., Athalye, A. & Mueller, J. (2021). *Pervasive label errors in test sets destabilize machine learning benchmarks.* NeurIPS Datasets and Benchmarks. [arXiv:2103.14749](https://arxiv.org/abs/2103.14749) · [labelerrors.com](https://labelerrors.com/)
- Rolnick, D. et al. (2017). *Deep learning is robust to massive label noise.* [arXiv:1705.10694](https://arxiv.org/abs/1705.10694)
- Sambasivan, N. et al. (2021). *"Everyone wants to do the model work, not the data work": Data cascades in high-stakes AI.* CHI. [doi:10.1145/3411764.3445518](https://doi.org/10.1145/3411764.3445518)
- Gebru, T. et al. (2021). *Datasheets for datasets.* Comm. ACM 64(12). [doi:10.1145/3458723](https://doi.org/10.1145/3458723)
- Zha, D. et al. (2023). *Data-centric artificial intelligence: A survey.* [arXiv:2303.10158](https://arxiv.org/abs/2303.10158)
- Buda, M., Maki, A. & Mazurowski, M. (2018). *A systematic study of the class imbalance problem in convolutional neural networks.* Neural Networks 106. [arXiv:1710.05381](https://arxiv.org/abs/1710.05381)
- Shorten, C. & Khoshgoftaar, T. (2019). *A survey on image data augmentation for deep learning.* J. Big Data 6. [doi:10.1186/s40537-019-0197-0](https://doi.org/10.1186/s40537-019-0197-0)
- Settles, B. (2009). *Active learning literature survey.* University of Wisconsin–Madison, TR 1648.
- Tools: [cleanlab](https://github.com/cleanlab/cleanlab) · [FiftyOne](https://docs.voxel51.com/) · [CVAT](https://www.cvat.ai/) · [Label Studio](https://labelstud.io/) · [DVC](https://dvc.org/)

**Data:** every image in this notebook is generated by `resources/scripts/partsim.py`.

## 17. Key takeaways
- Hold the model fixed and **engineer the data**; measure every change.
- **Systematic** label errors and noisy **test** labels are the dangerous ones; random noise is often tolerated.
- **Out-of-fold** predictions rank suspicious labels so that **human review** is efficient.
- Handle **imbalance** (collect, re-sample, re-weight; report per-class recall) and **coverage** (targeted augmentation, representative test sets).
- Audit for **near-duplicates / leakage** (thumbnail signatures, hashes, embeddings) and **box errors** before training; write a **labelling guideline**.

**Next session (09B):** training and deploying a custom detector on our conveyor data — ONNX, speed, and a C# corner.